# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** Every claim in this contract has a query cell under
it. A contract line without a query next to it is a guess.

Continues from `w02_ml_task_framing.ipynb`.

In [ ]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


## 1. Unit of analysis + time window

**One row = one pseudonymized content item** (`content_id`), summarising that page's performance over a
**single trailing 90-day window**. 30,000 rows, 32 clients, one row per page — verified below.

**The window, precisely.** The file carries **no date column at all**. There is no `report_date`, no
window start, no window end. Every time field is a *relative offset* measured from an unstated snapshot
date:

| Field | Window it covers |
|---|---|
| `*_90d` totals, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate` | the full trailing 90 days |
| `days_with_impressions`, `days_with_sessions` | count of active days inside those 90 |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | days 1–30 back (most recent 30) |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | days 31–60 back |
| `content_age_days`, `days_since_last_update` | offsets from the same unstated snapshot date |

Three window facts the queries below establish, each of which changes how I read the data:

1. **The two 30-day windows do not tile the 90-day window.** They cover days 1–60; days 61–90 are in the
   `*_90d` totals but in neither 30-day column. `last_30d + prev_30d == impressions_90d` holds for only
   **8.7%** of rows. So the label's comparison window is a *subset* of the feature window — I cannot
   treat the 90-day totals as "the same period" as the trend.
2. **`days_with_impressions` caps at 88, but `days_with_sessions` reaches 90.** Search data (GSC) is
   two days short of the analytics data (GA4) — the familiar GSC reporting lag, visible in the data
   rather than assumed. Any feature mixing the two is mixing windows of different length.
3. **The file is already filtered to mature pages.** `content_age_days` has a minimum of exactly **90**,
   so zero rows are younger than 90 days. This is why the "mature pages" filter in my ML-02 notebook
   dropped nothing: the filter was already applied upstream. Honest reading — my 30,000-row count is not
   the result of my own selection, it is the file as shipped.

**Consequence for the project.** Because there is no date, I cannot build a time-aware split from this
file, and I cannot define a past→future label. Both need the warehouse release
(`fact_content_daily_performance`, `report_date` 2025-01-27 → 2026-06-30). That is stated as a limit in
Section 4, not worked around.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# CLAIM 1: grain is one row per content_id.
dupes = df.groupby("content_id").size().pipe(lambda s: s[s > 1])
print(f"rows {len(df):,} | distinct content_id {df['content_id'].nunique():,} | duplicated {len(dupes)}")
assert len(dupes) == 0, "grain broken"

# CLAIM 2: there is no absolute date column anywhere in the file.
# Match whole name-tokens, so "days_since_last_update" (a relative offset) is not a false positive.
import re

DATE_TOKENS = {"date", "datetime", "timestamp", "ts", "day", "month", "year", "week"}
date_like = [c for c in df.columns if DATE_TOKENS & set(re.split(r"_+", c.lower()))]
print(f"absolute date columns found: {date_like or 'none'} -> no timestamp exists in this file")
assert date_like == [], "a date column exists after all - revisit the window section"
print("  (time is only ever a relative offset: days_since_last_update, content_age_days, days_with_*)")

# CLAIM 3: the 30-day windows do not tile the 90-day window.
tiles = (df["impressions_last_30d"] + df["impressions_prev_30d"] == df["impressions_90d"])
over = (df["impressions_last_30d"] + df["impressions_prev_30d"] > df["impressions_90d"]).sum()
print(f"last_30d + prev_30d == impressions_90d for {tiles.mean():.1%} of rows (over-count rows: {over})")
print("  -> days 61-90 sit in the 90d totals and in neither 30-day column")

# CLAIM 4: GSC window is 2 days shorter than GA4.
print(f"days_with_impressions: min {df['days_with_impressions'].min()} max {df['days_with_impressions'].max()}"
      f" | days_with_sessions max {df['days_with_sessions'].max()}")

# CLAIM 5: the file is pre-filtered to mature pages.
print(f"content_age_days: min {df['content_age_days'].min()} max {df['content_age_days'].max()}"
      f" | rows younger than 90 days: {(df['content_age_days'] < 90).sum()}")
print(f"days_since_last_update: min {df['days_since_last_update'].min()} max {df['days_since_last_update'].max()}")

## 2. Fields: feature / label / context / excluded

Every field I may touch goes in exactly one bucket: **44 columns from the file plus the one engineered
label = 45 classified**. The cell below asserts the totals match, so the contract cannot silently drift
from the file.

### Label and label-derived — never features (5)

| Field | Why it is out |
|---|---|
| `trend_direction` | The label source: `is_declining_label = (trend_direction == "down")`. |
| `trend_pct` | `trend_direction` is a threshold on this. Blank in 3,388 rows — exactly the rows where `impressions_prev_30d == 0`. |
| `impressions_last_30d` | With the field below, reconstructs the label **exactly** (measured 1.0000 in ML-03). |
| `impressions_prev_30d` | Same — the label is a deterministic function of this pair. |
| `is_declining_label` | The target itself (engineered in the setup cell). |

### Context — grouping, joining, splitting, reading; never learned from (2)

| Field | Role |
|---|---|
| `content_id` | The unit of analysis. Pseudonym. |
| `client_id` | Grouping key for the client-held-out split. Pseudonym. |

### Excluded on purpose (4)

| Field | Why |
|---|---|
| `clicks_last_30d`, `clicks_prev_30d` | Same last-vs-prev ratio shape as the label. Measured agreement with the label is only 0.5364 (near the 0.542 base rate), so they are *not* leaks — but they are weak, and I would rather drop a weak feature than defend a near-label ratio in review. |
| `sessions_last_30d`, `sessions_prev_30d` | Same reasoning; measured agreement 0.5383. |

`provider_used` (71.5% missing) and `model_used` (19.1% missing) are **generation-provenance /
product-decision flags**, not observed page performance. They describe which pipeline wrote the page.
Keeping them risks the model learning "which internal tool made this" rather than "is this page
decaying", and their missingness is itself a pipeline artifact. Both are excluded — that makes 6
excluded fields in total.

### Features — knowable at scoring time, safe to use (32)

- **Demand / market:** `search_volume`, `competition`, `competition_level`, `cpc`
- **Page shape:** `word_count`, `char_count`, `word_count_tier`, `char_count_tier`, `content_type`,
  `main_intent`
- **90-day performance totals:** `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`,
  `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`
- **Coverage / consistency:** `days_with_impressions`, `days_with_sessions`
- **Age and freshness:** `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`,
  `freshness_tier`
- **Rates and position:** `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- **Banded views:** `impression_tier`, `position_tier`

Two notes so a reviewer is not surprised: several features are **redundant with each other by
construction** (`position_tier` from `avg_position`, `impression_tier` from `impressions_90d`,
`*_tier` from their numeric parents). That is not leakage, it is collinearity — fine for trees, worth
naming for a linear model. And **`avg_position == 0` means "no data", not rank 0** (1,205 rows), so it
gets a missing flag rather than a zero.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The contract as code. Every column in the file lands in exactly one bucket.
LABEL_AND_DERIVED = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "is_declining_label",
]
CONTEXT = ["content_id", "client_id"]
EXCLUDED = [
    "clicks_last_30d", "clicks_prev_30d",      # same ratio shape as the label
    "sessions_last_30d", "sessions_prev_30d",  # same
    "provider_used", "model_used",             # product-decision / provenance flags
]
FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]

buckets = {"feature": FEATURES, "label": LABEL_AND_DERIVED, "context": CONTEXT, "excluded": EXCLUDED}
for name, cols in buckets.items():
    print(f"{name:9s} {len(cols):3d} fields")

classified = [c for cols in buckets.values() for c in cols]
assert len(classified) == len(set(classified)), "a field was put in two buckets"

unclassified = set(df.columns) - set(classified)
extra = set(classified) - set(df.columns)
print(f"\nfile columns {df.shape[1]} | classified {len(classified)}")
print(f"unclassified: {sorted(unclassified) or 'none'}")
print(f"named but absent from file: {sorted(extra) or 'none'}")
assert not unclassified, "every column must be classified"
assert not extra, "the contract names a column the file does not have"

# No feature may be label-derived. Belt and braces against future edits.
assert not set(FEATURES) & set(LABEL_AND_DERIVED), "a label-derived field leaked into FEATURES"
assert not set(FEATURES) & set(CONTEXT), "an ID leaked into FEATURES"
print("\nContract holds: 44/44 columns bucketed, no overlaps, no label-derived features.")

# avg_position == 0 means "no data" -> needs a flag, not a zero.
print(f"\navg_position == 0 (no data, NOT rank 0): {(df['avg_position'] == 0).sum():,} rows")

## 3. Verify it with queries (grain, counts, missing values, windows)

Grain and windows are checked in Section 1. This section checks the two things most likely to quietly
break a model: **missingness that follows a category**, and **rate columns on the wrong scale**.

**Missingness is not random — it tracks `content_type`.** Measured:

| Field | `feedly article` (n=2,096) | `keyword article` (n=27,207) | `comparison article` (n=697) |
|---|---|---|---|
| `search_volume`, `competition`, `cpc` | **100% missing** | 1.4% | 0% |
| `word_count`, `char_count` | 0% | **28.3% missing** | 0% |

**Why that is dangerous here, not just untidy.** The label rate differs sharply by `content_type`:
`feedly article` **28.7%** vs `keyword article` **56.1%** vs `comparison article` **57.2%**. So a blind
`fillna(0)` on `search_volume` would write a constant into exactly the 2,096 rows that are least likely
to be declining — handing the model a perfect proxy for "this is a feedly article" dressed up as a
demand feature. The fix in my pipeline: **`has_*` missing-indicator flags plus an explicit fill**, so
the model can use the missingness as the categorical signal it really is instead of inferring it.

The underlying cause is visible too: **46.6% of feedly articles are labelled `new`** (their
`impressions_prev_30d` is 0), against 4.6% of keyword articles. Feedly content is younger in
search terms, so it cannot be labelled `down`. That is a property of the label definition meeting the
content mix — not a decline signal.

**Rate columns are ×100 percentages, and two of them legitimately exceed 100.** Verified below:
`scroll_rate` exceeds 100 in **119** rows and `ai_traffic_pct` in **23** rows, because numerator and
denominator come from different measurement systems. These are not errors to clip. `ctr` reaching 100.0
is likewise 100%, not 100×.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# CLAIM: missingness follows content_type (it is patterned, not random).
print("Missing % by content_type:")
watch = ["search_volume", "competition", "cpc", "word_count", "char_count", "main_intent"]
miss_by_type = df.groupby("content_type")[watch].apply(lambda g: g.isna().mean() * 100).round(1)
print(miss_by_type.to_string())
print()

print("Rows and label rate by content_type (why patterned missingness matters):")
print(df.groupby("content_type")["is_declining_label"].agg(["size", "mean"]).round(3).to_string())
print()

print("Share labelled 'new' by content_type (prev-30d impressions == 0):")
print(pd.crosstab(df["content_type"], df["trend_direction"], normalize="index")["new"].round(3).to_string())
print()

print("Overall missing % (only columns with gaps):")
missing = (df.isna().mean() * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False).to_string())
print()

# CLAIM: trend_pct is blank exactly where impressions_prev_30d == 0.
blank_trend = df["trend_pct"].isna()
zero_prev = df["impressions_prev_30d"] == 0
print(f"trend_pct blank: {blank_trend.sum():,} | impressions_prev_30d == 0: {zero_prev.sum():,} | "
      f"identical rows: {(blank_trend == zero_prev).all()}")
print()

# CLAIM: rate columns are x100 percentages; scroll_rate and ai_traffic_pct may exceed 100.
print("Rate columns (x100 percentages - ctr 0.76 means 0.76%):")
for col in ("ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"):
    over = (df[col] > 100).sum()
    print(f"  {col:16s} max {df[col].max():7.2f}  rows > 100: {over}")
print("  -> scroll_rate / ai_traffic_pct over 100 are legitimate (different numerator/denominator systems)")

## 4. Data limits

What this data can never tell me, stated before anyone asks:

**1. It cannot support a forecast.** There is no date column and no future window, so every number is
concurrent. The honest sentence is "this page resembles the pages measured as declining", never "this
page will decline". A past→future label needs `fact_content_daily_performance` from the warehouse
release (`report_date` 2025-01-27 → 2026-06-30, per-client history depths that differ).

**2. It cannot support a causal claim about refreshing.** Nothing here records an intervention. No page
in this file was refreshed *because* of a score, and no outcome was measured after a refresh. Any
sentence of the form "refreshing these pages will recover traffic" is unsupported by this dataset and
would need a controlled or matched design. My output is a **review queue**, and that is the strongest
honest framing.

**3. The label is a definition, not an observed outcome.** `down` means a −20% impression change between
two 30-day windows. It is noise-sensitive for low-volume pages: a page going from 5 impressions to 3 is
"down −40%", identical in the label to a page going from 50,000 to 30,000. The label also cannot fire at
all for the 3,388 rows with `impressions_prev_30d == 0` (they are `new` or `flat`), which is why 46.6% of
feedly articles are structurally unlabelable as declining.

**4. It cannot speak about pages with no search data.** `avg_position == 0` in 1,205 rows means the page
has no position reading, not that it ranks first. Those rows are in the file but partly blind.

**5. It is a survivorship slice in two ways.** `content_age_days` starts at exactly 90, so pages younger
than 90 days were removed upstream; and every row has `impressions_90d > 0` and `sessions_90d > 0`, so
pages with zero traffic are absent entirely. Any statement like "x% of pages are declining" applies to
**mature pages that still get some traffic**, not to a content portfolio as a whole.

**6. The client panel is severely unbalanced.** 32 clients, from 3 pages to 7,008 pages, and per-client
label rates spanning **0.000 to 0.937**. Portfolio-wide averages are dominated by the largest few
clients, and any evaluation that does not hold clients out is measuring client memorisation.

**7. Windows are not aligned across systems.** GSC coverage caps at 88 days while GA4 reaches 90, and the
two 30-day columns cover only days 1–60 of the 90-day window. Features that mix them mix window
lengths — acceptable here because I keep them all as coarse features, but it forbids arithmetic like
"days 61–90 impressions = 90d total − last30 − prev30" being treated as a clean third window.

**Output, in one sentence.** The analysis hands a content strategist a ranked list of pages worth
opening first, each with a reason code and a confidence note — a prioritisation aid, not an automated
publishing decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Evidence for the limits section.
print("LIMIT 3 - the label is noise-sensitive at low volume.")
labelled_down = df[df["trend_direction"] == "down"]
print(f"  declining pages with < 100 impressions in 90d: "
      f"{(labelled_down['impressions_90d'] < 100).sum():,} of {len(labelled_down):,} "
      f"({(labelled_down['impressions_90d'] < 100).mean():.1%})")
print(f"  median impressions_90d among declining pages: {labelled_down['impressions_90d'].median():,.0f}")
print(f"  rows where the label cannot fire (impressions_prev_30d == 0): "
      f"{(df['impressions_prev_30d'] == 0).sum():,}")
print()

print("LIMIT 4 - pages with no position reading.")
print(f"  avg_position == 0 (no data): {(df['avg_position'] == 0).sum():,} rows "
      f"({(df['avg_position'] == 0).mean():.1%})")
print()

print("LIMIT 5 - survivorship: what was removed before I got the file.")
print(f"  min content_age_days: {df['content_age_days'].min()} (no pages younger than 90 days)")
print(f"  rows with impressions_90d == 0: {(df['impressions_90d'] == 0).sum()}")
print(f"  rows with sessions_90d == 0:    {(df['sessions_90d'] == 0).sum()}")
print("  -> 'x% of pages are declining' means: of mature pages that still get traffic.")
print()

print("LIMIT 6 - the client panel is unbalanced.")
per_client = df.groupby("client_id").agg(pages=("content_id", "size"), label_rate=("is_declining_label", "mean"))
print(per_client.describe().loc[["min", "50%", "max"]].round(3).to_string())
top3_share = per_client["pages"].nlargest(3).sum() / len(df)
print(f"  the 3 largest clients hold {top3_share:.1%} of all rows")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.